# 그림일기 생성 테스트 노트북

20개 테스트 케이스를 `/api/diary/generate` → `/api/diary/generate-image` 순서로 호출하고,  
결과는 `ai/eval_data/evaluations.jsonl`, `summary.csv`, `summary_pretty.csv`에 자동 저장됩니다.

**실행 전 확인사항**
- `main.py` 서버가 실행 중이어야 합니다 (`uvicorn main:app --reload`)
- API_URL이 서버 주소와 일치하는지 확인하세요

In [17]:
import requests
import json
import time
from datetime import date
from IPython.display import display, HTML

API_URL = "http://localhost:8001"
GENERATE_IMAGE = True   # 이미지 생성 + eval_data/images 저장 + summary 평가 기록
DELAY_SEC = 3           # 케이스 간 딜레이 (이미지 생성 포함이라 여유 있게)

print(f"서버 주소: {API_URL}")
print(f"이미지 생성: {'ON ✅  →  eval_data/images 저장 + summary 평가 기록' if GENERATE_IMAGE else 'OFF'}")

서버 주소: http://localhost:8001
이미지 생성: ON ✅  →  eval_data/images 저장 + summary 평가 기록


## 테스트 케이스 20세트 (2차)

| # | 견종 | 나이 | 상황 | 감정 | 일기유형 | 비고 |
|---|------|------|------|------|----------|------|
| 1 | 보더콜리 | 2살 | 공 물어오기 훈련 | 😊 | dog | 신규 |
| 2 | 닥스훈트 | 4살 | 계단 오르기 도전 | 🥹 | owner | 단신견종 |
| 3 | 불 테리어 | 3살 | 새벽 강변 나들이 | 😌 | daily | 야간 |
| 4 | 래브라도 리트리버 | 7살 | 유치원 방문 봉사 | 🤍 | memory | 노령 |
| 5 | 차우차우 | 2살 | 첫 목욕 | 😟 | dog | 신규 |
| 6 | 말리노이즈 | 5살 | 애견카페 | 😊 | daily | 신규 |
| 7 | 바셋 하운드 | 6살 | 낙엽밭 저녁 산책 | 😌 | owner | 야간 |
| 8 | 스피츠 | 1살 | 첫 눈 구경 | 🥹 | memory | 신규 |
| 9 | 아프간 하운드 | 3살 | 바람 부는 날 산책 | 😊 | dog | 신규 |
| 10 | 잭 러셀 테리어 | 4살 | 새 장난감 | 😊 | dog | 신규 |
| 11 | 보스턴 테리어 | 2살 | 생일파티 | 🥹 | memory | 신규 |
| 12 | 비숑 프리제 | 9살 | 할머니 댁 방문 | 🤍 | owner | 노령견 |
| 13 | 스코티시 테리어 | 5살 | 비 오는 날 | 😟 | daily | 야간 |
| 14 | 케언 테리어 | 3살 | 텃밭 탐험 | 😊 | dog | 신규 |
| 15 | 피레니안 마운틴 독 | 4살 | 첫 바다 | 🥹 | memory | 대형견 |
| 16 | 카발리에 킹 찰스 스패니얼 | 6살 | 음악 들으며 낮잠 | 😴 | owner | 야간 |
| 17 | 도베르만 | 3살 | 반려견 파크런 | 😊 | dog | 신규 |
| 18 | 뉴펀들랜드 | 2살 | 어린이 놀이터 | 🤍 | daily | 대형견 |
| 19 | 클럼버 스패니얼 | 5살 | 숲길 하이킹 | 🥹 | memory | 신규 |
| 20 | 라사압소 | 11살 | 밤 마지막 산책 | 😴 | owner | 노령견·야간 |

In [18]:
TEST_CASES = [
    {
        'no': 1,
        'pet_name': '모카',
        'breed': '웰시 코기',
        'breed_en': 'Welsh Corgi',
        'birth_date': '2023-09-12',
        'personalities': ['활발함', '식탐 있음'],
        'owner_name': '서준',
        'diary_type': 'owner',
        'emotion_emoji': '😂',
        'main_answers': [
            '아침에 양말을 찾는데 모카가 제 한 짝 양말을 입에 물고 거실을 신나게 도망다녔어요.',
            '잡히기 싫었는지 소파 주위를 빙글빙글 돌다가 중간에 저를 한번 쳐다보며 장난스럽게 꼬리를 흔들었어요.',
            '결국 간식이랑 교환했는데 양말보다 제가 뛰어다니는 반응이 더 재미있었던 것 같아 한참 웃었어요.'
        ]
    },
    {
        'no': 2,
        'pet_name': '리치',
        'breed': '포메라니안',
        'breed_en': 'Pomeranian',
        'birth_date': '2024-11-03',
        'personalities': ['겁 많음', '애교 많음'],
        'owner_name': '하린',
        'diary_type': 'memory',
        'emotion_emoji': '🥹',
        'main_answers': [
            '창문 밖에서 바람에 흔들리는 나뭇잎 그림자를 보고 리치가 한참 동안 조심스럽게 바라봤어요.',
            '바닥에 비친 움직이는 그림자가 신기하면서도 무서웠는지 제 다리 뒤로 숨었다가 다시 고개만 빼꼼 내밀었어요.',
            '나중에는 제 옆에 딱 붙어 앉아 안전하다고 느꼈는지 조용히 그림자 구경을 하는 모습이 참 사랑스러웠어요.'
        ]
    },
    {
        'no': 3,
        'pet_name': '호두',
        'breed': '시바 이누',
        'breed_en': 'Shiba Inu',
        'birth_date': '2022-01-18',
        'personalities': ['독립적', '표정 풍부함'],
        'owner_name': '예진',
        'diary_type': 'dog',
        'emotion_emoji': '😌',
        'main_answers': [
            '오늘은 창가에 앉아 밖을 지나는 사람들과 바람 소리를 천천히 구경했다.',
            '집사는 옆에 와서 말을 걸었지만 나는 굳이 대답하지 않고 세상 다 아는 얼굴로 창밖만 바라봤다.',
            '해가 조금 기울 무렵 따뜻한 빛이 등에 닿자 나는 조용히 눈을 감고 그 평화를 오래 즐겼다.'
        ]
    },
    {
        'no': 4,
        'pet_name': '루나',
        'breed': '사모예드',
        'breed_en': 'Samoyed',
        'birth_date': '2021-12-25',
        'personalities': ['사교적', '미소 많음'],
        'owner_name': '도윤',
        'diary_type': 'daily',
        'emotion_emoji': '😊',
        'main_answers': [
            '오늘은 루나와 넓은 잔디밭에서 공놀이를 했어요. 루나는 공보다 제가 던질 준비를 하는 순간부터 더 신나 했어요.',
            '공을 물어오면서도 입꼬리가 올라간 듯한 웃는 얼굴이 계속 보여서 보는 저까지 기분이 좋아졌어요.',
            '집에 돌아와 물을 한참 마신 뒤 바닥에 길게 누워 쉬는 모습까지도 행복해 보여서 뿌듯한 하루였어요.'
        ]
    },
    {
        'no': 5,
        'pet_name': '탱고',
        'breed': '미니어처 핀셔',
        'breed_en': 'Miniature Pinscher',
        'birth_date': '2023-07-07',
        'personalities': ['자신감 넘침', '경계심 강함'],
        'owner_name': '지후',
        'diary_type': 'owner',
        'emotion_emoji': '😅',
        'main_answers': [
            '청소기를 꺼내자 탱고가 오늘도 세상에서 제일 중요한 임무를 맡은 것처럼 앞에 딱 섰어요.',
            '가까이 다가가 으르렁거리다가도 청소기 머리가 움직이면 재빨리 뒤로 물러나는 모습이 너무 웃겼어요.',
            '청소가 끝나고 청소기를 세워두니 그제야 안심했는지 주변을 한 바퀴 돌며 자기 구역을 점검하더라고요.'
        ]
    },
    {
        'no': 6,
        'pet_name': '밤비',
        'breed': '비글',
        'breed_en': 'Beagle',
        'birth_date': '2020-10-09',
        'personalities': ['냄새에 민감', '장난기 많음'],
        'owner_name': '채원',
        'diary_type': 'memory',
        'emotion_emoji': '🤍',
        'main_answers': [
            '산책 중에 꽃집 앞을 지나는데 밤비가 갑자기 걸음을 늦추고 바닥 냄새를 아주 진지하게 맡기 시작했어요.',
            '꽃향기인지 지나간 강아지 냄새인지 모르겠지만 코를 바쁘게 움직이며 한참 동안 그 자리를 떠나지 않았어요.',
            '결국 제가 이름을 여러 번 부르자 아쉬운 얼굴로 돌아봤는데 그 모습이 꼭 탐정 같아서 웃음이 났어요.'
        ]
    },
    {
        'no': 7,
        'pet_name': '복실',
        'breed': '페키니즈',
        'breed_en': 'Pekingese',
        'birth_date': '2018-06-14',
        'personalities': ['느긋함', '자존심 셈'],
        'owner_name': '민지',
        'diary_type': 'dog',
        'emotion_emoji': '😴',
        'main_answers': [
            '오늘은 햇볕이 잘 드는 러그 한가운데를 내 자리로 정하고 느긋하게 시간을 보냈다.',
            '집사는 사진을 찍겠다며 앞에서 이름을 여러 번 불렀지만 나는 쉽게 시선을 주지 않았다.',
            '아주 잠깐만 올려다봐 주었더니 집사는 그 한 번의 눈맞춤만으로도 크게 감동한 얼굴이었다.'
        ]
    },
    {
        'no': 8,
        'pet_name': '마루',
        'breed': '진돗개',
        'breed_en': 'Jindo',
        'birth_date': '2021-04-02',
        'personalities': ['충성스러움', '침착함'],
        'owner_name': '현수',
        'diary_type': 'daily',
        'emotion_emoji': '🙂',
        'main_answers': [
            '저녁 무렵 한적한 길을 마루와 천천히 걸었어요. 오늘은 유난히 발걸음이 잘 맞아서 같이 걷는 리듬이 편안했어요.',
            '길가에서 잠깐 멈춰 서면 마루도 먼저 앞서가지 않고 제 옆에서 조용히 기다려줬어요.',
            '말은 없지만 늘 같은 속도로 함께 걸어주는 존재가 있다는 게 참 든든하게 느껴졌어요.'
        ]
    },
    {
        'no': 9,
        'pet_name': '젤리',
        'breed': '말티푸',
        'breed_en': 'Maltipoo',
        'birth_date': '2025-01-28',
        'personalities': ['호기심 많음', '애교 많음'],
        'owner_name': '소연',
        'diary_type': 'owner',
        'emotion_emoji': '🥰',
        'main_answers': [
            '오늘 젤리에게 처음으로 작은 계단을 보여줬어요. 처음에는 낯선 물건처럼 빙글빙글 돌며 확인만 했어요.',
            '제가 위에서 이름을 부르자 망설이다가 한 칸씩 조심스럽게 올라오는 모습이 너무 대견했어요.',
            '정상까지 올라온 뒤에는 금세 자신감이 붙었는지 내려갔다 올라오기를 반복하며 혼자 뿌듯해하더라고요.'
        ]
    },
    {
        'no': 10,
        'pet_name': '토리',
        'breed': '퍼그',
        'breed_en': 'Pug',
        'birth_date': '2019-08-30',
        'personalities': ['명랑함', '먹는 것 좋아함'],
        'owner_name': '승민',
        'diary_type': 'memory',
        'emotion_emoji': '😂',
        'main_answers': [
            '바닥에 작은 공을 굴려줬더니 토리가 진지한 얼굴로 쫓아가다가 갑자기 미끄러지듯 엎드렸어요.',
            '민망했는지 잠깐 멈춰 있더니 아무 일도 없었다는 표정으로 다시 공을 향해 돌진해서 더 웃겼어요.',
            '짧은 다리로 열심히 뛰어다니다가 결국 제 무릎 옆에 털썩 누워 숨을 고르는 모습이 오늘의 하이라이트였어요.'
        ]
    },
    {
        'no': 11,
        'pet_name': '별이',
        'breed': '요크셔테리어',
        'breed_en': 'Yorkshire Terrier',
        'birth_date': '2016-05-05',
        'personalities': ['예민함', '애정 깊음'],
        'owner_name': '은지',
        'diary_type': 'owner',
        'emotion_emoji': '🤍',
        'main_answers': [
            '비가 와서 창문을 조금 열어두었더니 별이가 제 무릎에 올라와 빗소리를 같이 들었어요.',
            '평소 작은 소리에도 예민한 편인데 오늘은 이상하게 차분한 얼굴로 창밖만 가만히 바라보더라고요.',
            '조용한 빗소리와 별이의 따뜻한 체온이 겹쳐져서 마음까지 천천히 가라앉는 기분이 들었어요.'
        ]
    },
    {
        'no': 12,
        'pet_name': '칩',
        'breed': '프렌치 불도그',
        'breed_en': 'French Bulldog',
        'birth_date': '2022-12-01',
        'personalities': ['장난기 많음', '사람 좋아함'],
        'owner_name': '가람',
        'diary_type': 'daily',
        'emotion_emoji': '😆',
        'main_answers': [
            '오늘 칩이 새로 산 삑삑 장난감을 처음 만났어요. 한 번 소리가 나자 눈이 동그래져서 장난감을 물고 바로 뒤로 물러났어요.',
            '조심스럽게 다시 앞발로 툭 건드려보다가 또 소리가 나자 신이 났는지 거실을 뛰어다니기 시작했어요.',
            '나중에는 스스로 삑삑 소리를 내며 혼자 가장 즐거운 파티를 여는 것 같았어요.'
        ]
    },
    {
        'no': 13,
        'pet_name': '오트',
        'breed': '시추',
        'breed_en': 'Shih Tzu',
        'birth_date': '2023-11-16',
        'personalities': ['온순함', '차분함'],
        'owner_name': '다인',
        'diary_type': 'dog',
        'emotion_emoji': '😊',
        'main_answers': [
            '오늘 나는 작은 쿠션 위에 앞발을 가지런히 모으고 오래 앉아 있었다.',
            '집사는 그런 내 모습이 너무 얌전하다며 계속 말을 걸었지만 나는 서두르지 않고 천천히 눈만 깜빡였다.',
            '조용한 오후의 공기와 부드러운 쿠션 감촉이 좋아서 한동안 그 자리를 떠나고 싶지 않았다.'
        ]
    },
    {
        'no': 14,
        'pet_name': '해피',
        'breed': '골든 리트리버',
        'breed_en': 'Golden Retriever',
        'birth_date': '2020-02-20',
        'personalities': ['온순함', '사교적'],
        'owner_name': '유나',
        'diary_type': 'memory',
        'emotion_emoji': '🥹',
        'main_answers': [
            '오늘은 해피와 동네를 산책하다가 어린아이를 만났어요. 아이가 조심스럽게 손을 내밀자 해피도 가만히 기다려줬어요.',
            '부드럽게 쓰다듬어도 놀라지 않고 천천히 꼬리만 흔드는 모습이 정말 믿음직스러웠어요.',
            '집으로 돌아오는 길에 괜히 제가 더 칭찬을 많이 하게 될 만큼 해피가 참 자랑스러웠어요.'
        ]
    },
    {
        'no': 15,
        'pet_name': '낭만',
        'breed': '보르조이',
        'breed_en': 'Borzoi',
        'birth_date': '2021-09-09',
        'personalities': ['우아함', '차분함'],
        'owner_name': '태희',
        'diary_type': 'daily',
        'emotion_emoji': '😌',
        'main_answers': [
            '커튼이 살짝 흔들리는 창가에 낭만이 길게 몸을 뉘이고 누워 있었어요.',
            '고개를 천천히 돌려 저를 바라보는 순간마저도 너무 우아해서 마치 한 장의 그림 같았어요.',
            '괜히 소리 내기 아까워 조용히 사진만 몇 장 찍었는데 그 평온한 분위기가 오래 기억에 남을 것 같아요.'
        ]
    },
    {
        'no': 16,
        'pet_name': '단추',
        'breed': '슈나우저',
        'breed_en': 'Schnauzer',
        'birth_date': '2022-05-27',
        'personalities': ['똑똑함', '경계심 강함'],
        'owner_name': '정민',
        'diary_type': 'owner',
        'emotion_emoji': '🙂',
        'main_answers': [
            '택배를 정리하려고 종이봉투를 바스락거리자 단추가 또 무슨 재미있는 일이 생긴 줄 알고 바로 달려왔어요.',
            '봉투 안을 고개만 쏙 넣어 확인하더니 원하는 게 없다는 걸 알자 금세 흥미를 잃는 표정이 너무 정확했어요.',
            '그래도 혹시 모르겠는지 제가 다른 봉투를 집을 때마다 다시 와서 검사하는 모습이 참 영리했어요.'
        ]
    },
    {
        'no': 17,
        'pet_name': '코코',
        'breed': '토이 푸들',
        'breed_en': 'Toy Poodle',
        'birth_date': '2024-06-06',
        'personalities': ['영리함', '활발함'],
        'owner_name': '라희',
        'diary_type': 'daily',
        'emotion_emoji': '😊',
        'main_answers': [
            '오늘은 코코에게 앉아, 기다려, 손을 순서대로 시켜봤어요. 코코는 눈을 반짝이며 제 손짓을 아주 집중해서 따라왔어요.',
            '특히 기다려를 할 때는 몸이 앞으로 나가고 싶은데도 꾹 참고 있는 표정이 너무 귀여웠어요.',
            '마지막에 칭찬을 해주니 꼬리를 빠르게 흔들며 스스로도 잘했다는 걸 아는 얼굴이었어요.'
        ]
    },
    {
        'no': 18,
        'pet_name': '흑임자',
        'breed': '차이니즈 크레스티드',
        'breed_en': 'Chinese Crested',
        'birth_date': '2021-11-11',
        'personalities': ['예민함', '애교 많음'],
        'owner_name': '세아',
        'diary_type': 'memory',
        'emotion_emoji': '🥹',
        'main_answers': [
            '오늘은 소파에 앉아 담요를 펼치자 흑임자가 어디선가 금방 달려와 제 옆에 몸을 붙였어요.',
            '부드러운 담요 끝을 코로 밀어 올리더니 안쪽으로 파고드는 모습이 너무 익숙해서 웃음이 났어요.',
            '이내 가장 포근한 자리를 찾은 듯 작은 숨소리를 내며 눈을 감는데 그 평화로운 얼굴이 참 예뻤어요.'
        ]
    },
    {
        'no': 19,
        'pet_name': '산들',
        'breed': '아이리시 세터',
        'breed_en': 'Irish Setter',
        'birth_date': '2023-03-03',
        'personalities': ['에너지 넘침', '명랑함'],
        'owner_name': '주원',
        'diary_type': 'dog',
        'emotion_emoji': '😄',
        'main_answers': [
            '오늘 나는 긴 줄을 달고 들판을 신나게 뛰었다.',
            '풀냄새와 바람이 한꺼번에 밀려와 몸이 저절로 앞으로 튀어나갔고 귀도 마음도 같이 날아가는 기분이었다.',
            '집사가 이름을 부를 때마다 나는 가장 신난 얼굴로 다시 돌아가 오늘의 즐거움을 몇 번이고 이어갔다.'
        ]
    },
    {
        'no': 20,
        'pet_name': '송이',
        'breed': '코커 스패니얼',
        'breed_en': 'Cocker Spaniel',
        'birth_date': '2017-01-09',
        'personalities': ['다정함', '감수성 풍부함'],
        'owner_name': '미래',
        'diary_type': 'owner',
        'emotion_emoji': '🤍',
        'main_answers': [
            '저녁에 조용히 정리를 하고 있는데 송이가 제 뒤를 따라다니다가 결국 제 옆자리에 털썩 앉았어요.',
            '가끔 올려다보는 눈빛이 너무 다정해서 하던 일을 멈추고 한참 머리를 쓰다듬어주게 됐어요.',
            '특별한 일은 없었지만 그런 평범한 순간 덕분에 오늘 하루가 유난히 포근하게 느껴졌어요.'
        ]
    }
]

In [19]:
# ── 일기 생성 함수 ─────────────────────────────────────────────────
def run_diary_generate(case: dict) -> dict | None:
    payload = {
        "pet_name": case["pet_name"],
        "breed": case["breed"],
        "breed_en": case.get("breed_en"),
        "birth_date": case.get("birth_date"),
        "personalities": case.get("personalities", []),
        "owner_name": case.get("owner_name", ""),
        "main_answers": case["main_answers"],
        "additional_answers": case.get("additional_answers", []),
        "diary_type": case["diary_type"],
        "emotion_emoji": case["emotion_emoji"],
    }
    try:
        res = requests.post(f"{API_URL}/api/diary/generate", json=payload, timeout=60)
        res.raise_for_status()
        return res.json()
    except Exception as e:
        print(f"  ❌ 일기 생성 실패: {e}")
        return None


def run_image_generate(image_prompt: str, session_id: str) -> bool:
    try:
        res = requests.post(
            f"{API_URL}/api/diary/generate-image",
            json={"image_prompt": image_prompt, "session_id": session_id},
            timeout=120,
        )
        res.raise_for_status()
        return True
    except Exception as e:
        print(f"  ❌ 이미지 생성 실패: {e}")
        return False

In [20]:
# ── 전체 테스트 실행 ───────────────────────────────────────────────
results = []

for case in TEST_CASES:
    no = case["no"]
    print(f"\n[{no:02d}/20] {case['breed']} / {case['pet_name']} / {case['emotion_emoji']}")
    print(f"  입력: {case['main_answers'][0][:40]}...")

    # 1단계: 일기 생성
    diary = run_diary_generate(case)
    if diary is None:
        results.append({"no": no, "status": "FAIL", "title": "-", "session_id": "-"})
        continue

    title = diary.get("title", "(제목 없음)")
    session_id = diary.get("session_id", "")
    print(f"  ✅ 제목: {title}")
    print(f"  📋 session_id: {session_id}")

    result = {
        "no": no,
        "breed": case["breed"],
        "pet_name": case["pet_name"],
        "emotion": case["emotion_emoji"],
        "diary_type": case["diary_type"],
        "title": title,
        "summary": diary.get("summary", ""),
        "session_id": session_id,
        "image_status": "SKIP",
    }

    # 2단계: 이미지 생성 (옵션)
    if GENERATE_IMAGE and diary.get("image_prompt"):
        print(f"  🎨 이미지 생성 중...")
        ok = run_image_generate(diary["image_prompt"], session_id)
        result["image_status"] = "OK" if ok else "FAIL"
        print(f"  이미지: {'✅' if ok else '❌'}")

    results.append(result)

    if no < len(TEST_CASES):
        time.sleep(DELAY_SEC)

print(f"\n{'='*50}")
print(f"완료: {sum(1 for r in results if r.get('title') != '-')}/{len(TEST_CASES)} 성공")
print(f"결과 저장 위치: ai/eval_data/evaluations.jsonl")


[01/20] 웰시 코기 / 모카 / 😂
  입력: 아침에 양말을 찾는데 모카가 제 한 짝 양말을 입에 물고 거실을 신나게 ...
  ✅ 제목: 양말은 내 친구!
  📋 session_id: 20260421_121200_283774
  🎨 이미지 생성 중...
  이미지: ✅

[02/20] 포메라니안 / 리치 / 🥹
  입력: 창문 밖에서 바람에 흔들리는 나뭇잎 그림자를 보고 리치가 한참 동안 조심...
  ✅ 제목: 그림자 구경 멍!
  📋 session_id: 20260421_121309_d477b0
  🎨 이미지 생성 중...
  이미지: ✅

[03/20] 시바 이누 / 호두 / 😌
  입력: 오늘은 창가에 앉아 밖을 지나는 사람들과 바람 소리를 천천히 구경했다....
  ✅ 제목: 창가에서 멍!
  📋 session_id: 20260421_121345_95a388
  🎨 이미지 생성 중...
  이미지: ✅

[04/20] 사모예드 / 루나 / 😊
  입력: 오늘은 루나와 넓은 잔디밭에서 공놀이를 했어요. 루나는 공보다 제가 던질...
  ✅ 제목: 신나는 공놀이!
  📋 session_id: 20260421_121429_4751a7
  🎨 이미지 생성 중...
  이미지: ✅

[05/20] 미니어처 핀셔 / 탱고 / 😅
  입력: 청소기를 꺼내자 탱고가 오늘도 세상에서 제일 중요한 임무를 맡은 것처럼 ...
  ✅ 제목: 청소기와의 대결!
  📋 session_id: 20260421_121512_83452c
  🎨 이미지 생성 중...
  이미지: ✅

[06/20] 비글 / 밤비 / 🤍
  입력: 산책 중에 꽃집 앞을 지나는데 밤비가 갑자기 걸음을 늦추고 바닥 냄새를 ...
  ✅ 제목: 꽃향기 탐정 놀이
  📋 session_id: 20260421_121554_3a7004
  🎨 이미지 생성 중...
  이미지: ✅

[07/20] 페키니즈 / 복실 / 😴
  입력: 오늘은 햇볕이 잘 드는 러그 한가운데를 내 자리로 정하고 느긋하게 시간을.

In [21]:
# ── 결과 요약 표 출력 ──────────────────────────────────────────────
rows = ""
for r in results:
    status_icon = "✅" if r.get("title") != "-" else "❌"
    rows += f"""
    <tr>
        <td>{r['no']}</td>
        <td>{r.get('breed', '-')}</td>
        <td>{r.get('pet_name', '-')}</td>
        <td>{r.get('emotion', '-')}</td>
        <td>{r.get('diary_type', '-')}</td>
        <td>{status_icon} {r.get('title', '-')}</td>
        <td>{r.get('summary', '-')}</td>
        <td style='font-size:11px;color:#888'>{r.get('session_id', '-')[:20]}...</td>
    </tr>"""

html = f"""
<style>table{{border-collapse:collapse;width:100%}} th,td{{border:1px solid #ddd;padding:6px 10px;font-size:13px}} th{{background:#f5f5f5}}</style>
<table>
  <tr>
    <th>#</th><th>견종</th><th>이름</th><th>감정</th><th>유형</th>
    <th>생성 제목</th><th>한줄 요약</th><th>session_id</th>
  </tr>
  {rows}
</table>
"""
display(HTML(html))

#,견종,이름,감정,유형,생성 제목,한줄 요약,session_id
1,웰시 코기,모카,😂,owner,✅ 양말은 내 친구!,"양말 놀이, 서준과 즐거운 아침!",20260421_121200_2837...
2,포메라니안,리치,🥹,memory,✅ 그림자 구경 멍!,그림자 구경하며 용기 냈어요!,20260421_121309_d477...
3,시바 이누,호두,😌,dog,✅ 창가에서 멍!,창가에서 평화로운 하루!,20260421_121345_95a3...
4,사모예드,루나,😊,daily,✅ 신나는 공놀이!,잔디밭에서 신나는 공놀이!,20260421_121429_4751...
5,미니어처 핀셔,탱고,😅,owner,✅ 청소기와의 대결!,청소기와의 신나는 대결!,20260421_121512_8345...
6,비글,밤비,🤍,memory,✅ 꽃향기 탐정 놀이,꽃집 앞 탐정 놀이!,20260421_121554_3a70...
7,페키니즈,복실,😴,dog,✅ 햇살 속 졸음,햇살 속에서 졸린 하루,20260421_121635_2e1a...
8,진돗개,마루,🙂,daily,✅ 현수랑 나란히,현수랑 나란히 걷기!,20260421_121715_2618...
9,말티푸,젤리,🥰,owner,✅ 계단 탐험 대성공!,계단 정복! 뿌듯한 하루!,20260421_121754_d2b2...
10,퍼그,토리,😂,memory,✅ 공이랑 미끄럼 놀이,공놀이하다 미끄러졌어요!,20260421_121833_b6cf...


In [22]:
# ── (선택) 특정 케이스 단독 실행 ───────────────────────────────────
# 특정 번호만 재테스트하고 싶을 때 케이스 번호를 바꾸세요

SINGLE_NO = 1  # 1 ~ 20

case = next(c for c in TEST_CASES if c["no"] == SINGLE_NO)
print(f"[단독 실행] #{SINGLE_NO} {case['breed']} / {case['pet_name']}")

diary = run_diary_generate(case)
if diary:
    print(f"\n제목   : {diary.get('title')}")
    print(f"요약   : {diary.get('summary')}")
    print(f"본문   :\n{diary.get('content')}")
    print(f"\nsession_id: {diary.get('session_id')}")

[단독 실행] #1 웰시 코기 / 모카

제목   : 양말은 내 친구!
요약   : 양말 놀이가 최고야!
본문   :
오늘 아침, 나는 서준의 양말 한 짝을 입에 물고 거실을 뛰어다녔어! 멍! 서준이 양말을 찾느라 바빴는데, 나는 그게 너무 재미있었어. 소파 주위를 빙글빙글 돌면서 서준을 살짝 쳐다봤지. 그때 서준이 웃으면서 나를 쫓아왔어. 결국 간식이랑 양말을 바꿨는데, 사실 양말보다 서준이랑 놀던 게 더 즐거웠어. 서준이 웃는 소리가 정말 기분 좋았어. 오늘도 서준이랑 함께해서 행복해!

session_id: 20260421_122558_83e561
